### Setup envs

In [1]:
import os
import logging
import time

import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_recommenders as tfrs

### Model definition

In [2]:
class UserModel(tf.keras.Model):

    def __init__(self, conf):
        super().__init__()

        self.gender_embedding = tf.keras.Sequential([
            tf.keras.layers.experimental.preprocessing.StringLookup(
                vocabulary=conf['unique_genders'], mask_token=None),
            tf.keras.layers.Embedding(len(conf['unique_genders']) + 1, 4),
        ])

        self.lang_embedding = tf.keras.Sequential([
            tf.keras.layers.experimental.preprocessing.StringLookup(
                vocabulary=conf['unique_langs'], mask_token=None),
            tf.keras.layers.Embedding(len(conf['unique_langs']) + 1, 10),
        ])

        self.country_embedding = tf.keras.Sequential([
            tf.keras.layers.experimental.preprocessing.StringLookup(
                vocabulary=conf['unique_countries'], mask_token=None),
            tf.keras.layers.Embedding(len(conf['unique_countries']) + 1, 10),
        ])

        self.network_embedding = tf.keras.Sequential([
            tf.keras.layers.experimental.preprocessing.StringLookup(
                vocabulary=conf['unique_networks'], mask_token=None),
            tf.keras.layers.Embedding(len(conf['unique_networks']) + 1, 4),
        ])

        age_boundaries = np.array(conf['age_boundaries'])
        self.viewer_age_embedding = tf.keras.Sequential([
            tf.keras.layers.experimental.preprocessing.Discretization(age_boundaries.tolist()),
            tf.keras.layers.Embedding(len(age_boundaries), 2)
        ])

        self.centroids = tf.constant(conf['centroids'])
        self.viewer_lat_long_embedding = tf.keras.Sequential([
            tf.keras.layers.experimental.preprocessing.TextVectorization(
                standardize = None, split = self.classify,
                vocabulary = [str(i) for i in range(len(self.centroids))],
                max_tokens=len(self.centroids) + 2
                ),
            tf.keras.layers.Embedding(len(self.centroids) + 2, 2),
        ])

    @tf.function()
    def call(self, inputs):
        return tf.concat([
            self.gender_embedding(inputs["viewer_gender"]),
            self.lang_embedding(inputs["viewer_lang"]),
            self.country_embedding(inputs["viewer_country"]),
            self.network_embedding(inputs["viewer_network"]),
            self.viewer_age_embedding(inputs["viewer_age"]),
            self.viewer_lat_long_embedding(inputs["viewer_lat_long"]),
        ], axis = 1)

    @tf.keras.utils.register_keras_serializable()
    def classify(self, pair):
        """
        given a datapoint, compute the cluster closest to the datapoint. Return the cluster ID of that cluster.
        :param pair:
        :return: cluster ID
        """
        str_data = tf.strings.split(pair, sep = ",").values
        str_data = tf.map_fn(lambda x: tf.strings.regex_replace(x, "b'", ""), str_data)
        datapoints = tf.map_fn(lambda x: tf.strings.to_number(x), str_data, dtype = (tf.float32))
        datapoints = tf.reshape(datapoints, [-1, 2])

        expanded_centroids = tf.expand_dims(self.centroids, 1)
        expanded_vectors = tf.expand_dims(datapoints, 0)
        distances = tf.reduce_sum(tf.square(tf.subtract(expanded_vectors, expanded_centroids)), 2)
        clusters = tf.math.argmin(distances)
        return tf.strings.as_string(clusters)

In [3]:
class QueryModel(tf.keras.Model):

	def __init__(self, conf):
		super().__init__()

		# We first use the user model for generating embeddings.
		self.embedding_model = UserModel(conf)
		self.dense_layers = tf.keras.Sequential(
			[
				tf.keras.layers.Dense(32, activation = 'relu', kernel_regularizer = tf.keras.regularizers.L2(0.0001)),
				tf.keras.layers.Dense(32)
			]
		)

	def call(self, inputs):
		feature_embedding = self.embedding_model(inputs)
		return self.dense_layers(feature_embedding)

In [4]:
class BroadcasterModel(tf.keras.Model):

    def __init__(self, conf):
        super().__init__()

        self.broadcaster_embedding = tf.keras.Sequential([
            tf.keras.layers.experimental.preprocessing.StringLookup(
                vocabulary=conf['unique_broadcasters'], mask_token=None),
            tf.keras.layers.Embedding(len(conf['unique_broadcasters']) + 1, conf['broadcaster_embedding_dimension'])
        ])

    def call(self, broadcaster):
        return tf.concat([
            self.broadcaster_embedding(broadcaster),
        ], axis=1)

In [5]:
class CandidateModel(tf.keras.Model):

	def __init__(self, conf):
		super().__init__()

		self.embedding_model = BroadcasterModel(conf)

		self.dense_layers = tf.keras.Sequential(
			[
				tf.keras.layers.Dense(32, activation = 'relu', kernel_regularizer = tf.keras.regularizers.L2(0.0001)),
				tf.keras.layers.Dropout(0.5),
				tf.keras.layers.Dense(32)
			]
		)

	def call(self, inputs):
		feature_embedding = self.embedding_model(inputs)
		return self.dense_layers(feature_embedding)

### Load data

In [6]:
def load_data_file_cold(file, stats):
    print('loading file:' + file)
    training_df = pd.read_csv(
        file,
        skiprows=[0],
        names=["viewer",
               "broadcaster",
               "viewer_age",
               "viewer_gender",
               "viewer_longitude",
               "viewer_latitude",
               "viewer_lang",
               "viewer_country",
               "broadcaster_age",
               "broadcaster_gender",
               "broadcaster_longitude",
               "broadcaster_latitude",
               "broadcaster_lang",
               "broadcaster_country",
               "duration", 
               "viewer_network", 
               "broadcaster_network", 
               "count"], 
        dtype={
            'viewer': np.unicode,
            'broadcaster': np.unicode,
            'viewer_age': np.single,
            'viewer_gender': np.unicode,
            'viewer_longitude': np.single,
            'viewer_latitude': np.single,
            'viewer_lang': np.unicode,
            'viewer_country': np.unicode,
            'broadcaster_age': np.single,
            'broadcaster_longitude': np.single,
            'broadcaster_latitude': np.single,
            'broadcaster_lang': np.unicode,
            'broadcaster_country': np.unicode,
            'duration': np.single,
            'viewer_network': np.unicode,
            'broadcaster_network': np.unicode,
            'count': np.unicode,
        })

    values = {
        'viewer': 'unknown',
        'broadcaster': 'unknown',
        'viewer_age': 30,
        'viewer_gender': 'unknown',
        'viewer_longitude': 0,
        'viewer_latitude': 0,
        'viewer_lang': 'unknown',
        'viewer_country': 'unknown',
        'broadcaster_age': 30,
        'broadcaster_longitude': 0,
        'broadcaster_latitude': 0,
        'broadcaster_lang': 'unknown',
        'broadcaster_country': 'unknown',
        'duration': 0,
        'viewer_network': 'unknown',
        'broadcaster_network': 'unknown',
        "viewer_lat_long": tf.constant(["40.36393,-74.89611"]),
        'count': '1'
    }

    training_df = training_df.sample(frac=0.1)
    training_df.fillna(value=values, inplace=True)
    training_df['viewer_lat_long'] = training_df[['viewer_latitude', 'viewer_longitude']].apply(lambda x: '{},{}'.format(x[0],x[1]), axis=1)
    training_df['duration'] = np.log(1 + training_df['duration'])
    print(training_df.head(10))
    print(training_df.iloc[-10:])
    # stats.send_stats('data-size', len(training_df.index))
    return training_df


def load_training_data_cold(file, stats):
    ratings_df = load_data_file_cold(file, stats)
    print('creating data set')
    training_ds = (
        tf.data.Dataset.from_tensor_slices(
            ({
                "viewer": tf.cast(
                    ratings_df['viewer'].values,
                    tf.string),
                "viewer_gender": tf.cast(
                    ratings_df['viewer_gender'].values,
                    tf.string),
                "viewer_lang": tf.cast(
                    ratings_df['viewer_lang'].values,
                    tf.string),
                "viewer_country": tf.cast(
                    ratings_df['viewer_country'].values,
                    tf.string),
                "viewer_age": tf.cast(
                    ratings_df['viewer_age'].values,
                    tf.int32),
                "viewer_longitude": tf.cast(
                    ratings_df['viewer_longitude'].values,
                    tf.float16),
                "viewer_latitude": tf.cast(
                    ratings_df['viewer_latitude'].values,
                    tf.float16),
                "broadcaster": tf.cast(
                    ratings_df['broadcaster'].values,
                    tf.string),
                "viewer_network": tf.cast(
                    ratings_df['viewer_network'].values,
                    tf.string),
                "broadcaster_network": tf.cast(
                    ratings_df['broadcaster_network'].values,
                    tf.string),
                "duration": tf.cast(
                    ratings_df['duration'].values,
                    tf.float16),
                "viewer_lat_long": tf.cast(
                    ratings_df['viewer_lat_long'].values,
                    tf.string),
            })))

    return training_ds
            

def prepare_training_data_cold(train_ds):
    print('prepare_training_data')
    training_ds = train_ds.cache().map(lambda x: {
        "broadcaster": x["broadcaster"],
        "viewer": x["viewer"],
        "viewer_gender": x["viewer_gender"],
        "viewer_lang": x["viewer_lang"],
        "viewer_country": x["viewer_country"],
        "viewer_age": x["viewer_age"],
        "viewer_longitude": x["viewer_longitude"],
        "viewer_latitude": x["viewer_latitude"],
        "viewer_network": x["viewer_network"],
        "broadcaster_network": x["broadcaster_network"],
        "duration": x["duration"],
        "viewer_lat_long": x["viewer_lat_long"],
    }, num_parallel_calls=tf.data.AUTOTUNE,
       deterministic=False)

    print('done prepare_training_data')
    return training_ds

In [7]:
def get_broadcaster_data_set(train_ds):
    broadcasters = train_ds.cache().map(lambda x: x["broadcaster"], num_parallel_calls=tf.data.AUTOTUNE, deterministic=False)
    broadcasters_ds = tf.data.Dataset.from_tensor_slices(
        np.unique(list(broadcasters.as_numpy_iterator())))
    return broadcasters_ds

In [8]:
training_dataset = load_training_data_cold("csv/2022-01-12.csv", "")

loading file:csv/2022-01-12.csv
                                                  viewer  \
3977378  29 11 55 50 6c 22 af 9e d1 8e a5 78 f7 1a d4 27   
7831980  d3 44 32 4b c3 fa 44 c3 22 be e9 f0 f4 5e f1 60   
4760718  02 19 d2 69 1c 84 9e 2b 0b 2d 3b 07 34 df 0c bf   
6912373  8d c2 ac 09 32 c9 a6 d3 0e ec b4 59 4a 24 68 7b   
7918140  b0 52 29 d6 e8 58 02 d7 86 d5 e2 a4 72 33 b8 9f   
6998910  c2 ea 6f 12 f5 99 9d 61 39 39 0a f6 f1 73 96 a6   
1599986  be aa 2d 6c b8 37 3e 49 9a 28 30 83 c2 e4 f1 86   
5034264  ee 14 eb a7 6b 29 e9 2c ca 7e 1e d6 a0 25 01 62   
6117946  ca e6 55 8b 04 3f e2 39 34 e5 e6 04 37 14 e1 82   
7605984  89 00 78 a1 53 93 5a bb 18 a4 6c 18 68 d5 bc c4   

                                             broadcaster  viewer_age  \
3977378  8a 11 85 37 a3 39 18 99 3a 78 3b 13 12 6e 41 7e        32.0   
7831980  fb ba 91 dd 29 b6 66 17 1a 17 26 e4 d6 ec bf a0        33.0   
4760718  66 22 c4 c3 ee c5 ca 20 90 e0 3b ac 9c 4e 0b 54        29.0   
6912373  ea b3 e3 1

In [9]:
train = prepare_training_data_cold(training_dataset)

prepare_training_data
done prepare_training_data


In [10]:
broadcasters_data_set = get_broadcaster_data_set(training_dataset)

### Prepare model conf

In [11]:
def get_list(training_data, key):
    return training_data.batch(1_000_000).map(lambda x: x[key], num_parallel_calls=tf.data.AUTOTUNE, deterministic=False)


def get_unique_list(data):
    return np.unique(np.concatenate(list(data)))

In [12]:
user_genders = get_list(train, 'viewer_gender')

In [13]:
user_langs = get_list(train, 'viewer_lang')

In [14]:
user_countries = get_list(train, 'viewer_country')

In [15]:
viewer_age = get_list(train, 'viewer_age')

In [16]:
user_networks = get_list(train, 'viewer_network')

### derive input dims

In [17]:
unique_user_genders = get_unique_list(user_genders)

In [18]:
len(unique_user_genders)

3

In [19]:
unique_user_langs = get_unique_list(user_langs)

In [20]:
len(unique_user_langs)

62

In [21]:
unique_user_countries = get_unique_list(user_countries)

In [22]:
len(unique_user_countries)

176

In [23]:
unique_user_networks = get_unique_list(user_networks)

In [24]:
len(unique_user_networks)

5

In [25]:
broadcaster_ids = get_list(train, 'broadcaster')

In [26]:
unique_broadcasters = get_unique_list(broadcaster_ids)

In [27]:
len(unique_broadcasters)

81267

In [28]:
broadcaster_embedding_dimension = 32

In [29]:
cold_start_conf = {
    'unique_genders': unique_user_genders,
    'unique_langs': unique_user_langs,
    'unique_countries': unique_user_countries,
    'unique_networks': unique_user_networks,
    'unique_broadcasters': unique_broadcasters,
    'broadcaster_embedding_dimension': broadcaster_embedding_dimension,
    'age_boundaries': [18, 25, 30, 35, 40, 45, 50, 55, 60, 65, float("inf")],
    'centroids': [[36.68147669256268, -82.8910274009993],
        [23.22243322909555, 78.23027450833709],
        [50.04997682638993, 0.22379313938744885],
        [37.9309447099281, -117.00741350764692],
        [-32.795864819917725, 148.7159172660312],
        [-18.570548393114084, -54.280255665692565],
        [13.921140442819565, 116.38740315555172],
        [29.78951080730802, 40.279515865947936]]
}

In [30]:
cold_start_conf

{'unique_genders': array([b'female', b'male', b'unknown'], dtype=object),
 'unique_langs': array([b'ar', b'az', b'bg', b'bm', b'bn', b'bs', b'ca', b'co', b'cs',
        b'da', b'de', b'el', b'en', b'es', b'et', b'fa', b'fi', b'fr',
        b'gl', b'gu', b'he', b'hi', b'hr', b'hu', b'id', b'in', b'it',
        b'iw', b'ja', b'ka', b'ko', b'lo', b'lt', b'lv', b'mk', b'ml',
        b'mr', b'ms', b'nb', b'ne', b'nl', b'ny', b'pa', b'pl', b'ps',
        b'pt', b'ro', b'ru', b'sk', b'sm', b'sq', b'sr', b'sv', b'ta',
        b'te', b'th', b'ti', b'tr', b'ur', b'uz', b'vi', b'zh'],
       dtype=object),
 'unique_countries': array([b'419', b'AC', b'AD', b'AE', b'AF', b'AG', b'AL', b'AO', b'AQ',
        b'AR', b'AS', b'AT', b'AU', b'AW', b'AX', b'AZ', b'BA', b'BD',
        b'BE', b'BF', b'BG', b'BH', b'BJ', b'BN', b'BO', b'BQ', b'BR',
        b'BS', b'BT', b'BY', b'BZ', b'CA', b'CF', b'CH', b'CI', b'CL',
        b'CM', b'CN', b'CO', b'CR', b'CV', b'CY', b'CZ', b'DE', b'DK',
        b'DO', b'DZ',

### query model

In [31]:
query_model = QueryModel(cold_start_conf)

### broadcaster model

In [32]:
candidate_model = CandidateModel(cold_start_conf)

### Candidate / Ranking model

In [33]:
class RankingModel(tf.keras.Model):

	def __init__(self):
		super().__init__()
		embedding_dimension = 32

		# Compute predictions.
		self.ratings = tf.keras.Sequential(
			[
				# Learn multiple dense layers.
				tf.keras.layers.Dense(256, activation = "relu"),
				tf.keras.layers.Dense(64, activation = "relu"),
				# Make rating predictions in the final layer.
				tf.keras.layers.Dense(1)
			]
		)

	def call(self, inputs):
		query_embeddings, positive_broadcaster_embeddings = inputs
		return self.ratings(tf.concat([query_embeddings, positive_broadcaster_embeddings], axis = 1))

In [34]:
ranking_model = RankingModel()

### Loss and metrics

In [35]:
ranking_task = tfrs.tasks.Ranking(
  loss = tf.keras.losses.MeanSquaredError(),
  metrics=[tf.keras.metrics.RootMeanSquaredError()]
)

In [36]:
retrieval_task = tfrs.tasks.Retrieval(
    metrics=tfrs.metrics.FactorizedTopK(
        candidates=broadcasters_data_set.batch(128).map(candidate_model)
    )
)

In [37]:
from typing import Dict, Text

In [38]:
class TwoTowers(tfrs.models.Model) :

    def __init__(self, candidate_model, query_model, ranking_model, ranking_task, retrieval_task, ranking_weight, retrieval_weight):
        super().__init__()
        
        self.query_model: tf.keras.Model = query_model
        self.candidate_model: tf.keras.Model = candidate_model
        self.ranking_model: tf.keras.Model = ranking_model
        self.ranking_task = ranking_task
        self.retrieval_task = retrieval_task
        
        # The loss weights.
        self.ranking_weight = ranking_weight
        self.retrieval_weight = retrieval_weight
    
    def train_step(self, features: Dict[Text, tf.Tensor]) -> tf.Tensor:
        with tf.GradientTape() as tape:
            query_embeddings = self.query_model({
                "viewer_gender": features["viewer_gender"],
                "viewer_lang": features["viewer_lang"],
                "viewer_country": features["viewer_country"],
                "viewer_age": features["viewer_age"],
                "viewer_network": features["viewer_network"],
                "viewer_latitude": features["viewer_latitude"],
                "viewer_longitude": features["viewer_longitude"],
                "viewer_lat_long": features["viewer_lat_long"],
            })
            positive_broadcaster_embeddings = self.candidate_model(
                features["broadcaster"])
            
            labels = features["duration"]
            ranking_predictions = self.ranking_model(
                (query_embeddings, positive_broadcaster_embeddings)
            )
            ranking_loss = self.ranking_task(
                labels = labels,
                predictions = ranking_predictions,
            )
            
            retrieval_loss = self.retrieval_task(query_embeddings, positive_broadcaster_embeddings)
            
            regularization_loss = sum(self.losses)
            
            total_loss = regularization_loss + self.ranking_weight * ranking_loss + self.retrieval_weight * retrieval_loss
        
        gradients = tape.gradient(total_loss, self.trainable_variables)
        self.optimizer.apply_gradients(
            zip(gradients, self.trainable_variables)
        )
        
        metrics = {metric.name: metric.result() for metric in self.metrics}
        metrics["loss"] = ranking_loss 
        metrics["regularization_loss"] = regularization_loss
        metrics["total_loss"] = total_loss
        
        return metrics

    def test_step(self, features: Dict[Text, tf.Tensor]) -> tf.Tensor:
        labels = features["duration"]
        
        query_embeddings = self.query_model({
            "viewer_gender": features["viewer_gender"],
            "viewer_lang": features["viewer_lang"],
            "viewer_country": features["viewer_country"],
            "viewer_age": features["viewer_age"],
            "viewer_network": features["viewer_network"],
            "viewer_latitude": features["viewer_latitude"],
            "viewer_longitude": features["viewer_longitude"],
            "viewer_lat_long": features["viewer_lat_long"],
        })
        positive_broadcaster_embeddings = self.candidate_model(
            features["broadcaster"])
        
        rating_predictions = self.ranking_model(
            (query_embeddings, positive_broadcaster_embeddings)
        )
        
        retrieval_loss = self.retrieval_task(query_embeddings, positive_broadcaster_embeddings)

        # The task computes the loss and the metrics.
        ranking_loss = self.ranking_task(labels = labels, predictions = rating_predictions)
        
        regularization_loss = sum(self.losses)
        
        total_loss = regularization_loss + self.ranking_weight * ranking_loss + self.retrieval_weight * retrieval_loss
        
        metrics = {metric.name: metric.result() for metric in self.metrics}
        metrics["loss"] = ranking_loss + retrieval_loss
        metrics["regularization_loss"] = regularization_loss
        metrics["total_loss"] = total_loss
        return metrics        

In [39]:
ranking_weight = 1
retrieval_weight = 1

In [40]:
model = TwoTowers(candidate_model, query_model, ranking_model, ranking_task, retrieval_task, ranking_weight, retrieval_weight)

In [48]:
learning_rate = 0.1
batch_size = 16384
epochs = 30
patience = 3
top_k = 1999

In [49]:
model.compile(optimizer=tf.keras.optimizers.Adagrad(learning_rate=learning_rate))

In [50]:
tf.random.set_seed(42)
shuffled = train.shuffle(100_000, seed=42, reshuffle_each_iteration=True)

train_p80 = shuffled.take(80_000)
test_p20 = shuffled.skip(80_000).take(20_000)

cached_train = train_p80.shuffle(100_000).batch(batch_size)
cached_test = test_p20.batch(batch_size).cache()

In [51]:
# model.fit(train_ds, epochs=epochs)
callback = tf.keras.callbacks.EarlyStopping(monitor = "total_loss", 
    patience = patience,
    verbose=1,
    restore_best_weights=True)
hist = model.fit(
    cached_train,
    validation_data=cached_test,
    validation_freq=1,
    epochs=epochs,
    callbacks = [callback])

Epoch 1/30
5/5 [==============================] - 255s 54s/step - root_mean_squared_error: 6.2811 - factorized_top_k/top_1_categorical_accuracy: 0.0017 - factorized_top_k/top_5_categorical_accuracy: 0.0077 - factorized_top_k/top_10_categorical_accuracy: 0.0127 - factorized_top_k/top_50_categorical_accuracy: 0.0390 - factorized_top_k/top_100_categorical_accuracy: 0.0617 - loss: 35.2211 - regularization_loss: 0.0150 - total_loss: 260932.1510 - val_root_mean_squared_error: 2.8431 - val_factorized_top_k/top_1_categorical_accuracy: 4.0000e-04 - val_factorized_top_k/top_5_categorical_accuracy: 0.0016 - val_factorized_top_k/top_10_categorical_accuracy: 0.0033 - val_factorized_top_k/top_50_categorical_accuracy: 0.0138 - val_factorized_top_k/top_100_categorical_accuracy: 0.0265 - val_loss: 27753.7793 - val_regularization_loss: 0.0143 - val_total_loss: 27753.7930
Epoch 2/30
5/5 [==============================] - 159s 33s/step - root_mean_squared_error: 3.2353 - factorized_top_k/top_1_categorical

5/5 [==============================] - 180s 36s/step - root_mean_squared_error: 1.4446 - factorized_top_k/top_1_categorical_accuracy: 0.0041 - factorized_top_k/top_5_categorical_accuracy: 0.0149 - factorized_top_k/top_10_categorical_accuracy: 0.0225 - factorized_top_k/top_50_categorical_accuracy: 0.0536 - factorized_top_k/top_100_categorical_accuracy: 0.0767 - loss: 2.0838 - regularization_loss: 0.0152 - total_loss: 129275.4948 - val_root_mean_squared_error: 1.4763 - val_factorized_top_k/top_1_categorical_accuracy: 0.0012 - val_factorized_top_k/top_5_categorical_accuracy: 0.0055 - val_factorized_top_k/top_10_categorical_accuracy: 0.0093 - val_factorized_top_k/top_50_categorical_accuracy: 0.0336 - val_factorized_top_k/top_100_categorical_accuracy: 0.0580 - val_loss: 25391.3516 - val_regularization_loss: 0.0153 - val_total_loss: 25391.3672
Epoch 20/30
5/5 [==============================] - 189s 38s/step - root_mean_squared_error: 1.4877 - factorized_top_k/top_1_categorical_accuracy: 0.00

In [52]:
hist.history

{'root_mean_squared_error': [6.28108024597168,
  3.2352912425994873,
  1.5847362279891968,
  1.4926221370697021,
  1.4948066473007202,
  1.5006848573684692,
  1.4794946908950806,
  1.4466763734817505,
  1.4549747705459595,
  1.5188394784927368,
  1.500477910041809,
  1.4661113023757935,
  1.449584722518921,
  1.4629660844802856,
  1.4606852531433105,
  1.4480245113372803,
  1.4444496631622314,
  1.4445900917053223,
  1.4445546865463257,
  1.487674593925476,
  1.4612884521484375,
  1.4616820812225342,
  1.4503729343414307,
  1.442185401916504,
  1.4383471012115479,
  1.4440679550170898,
  1.4671919345855713],
 'factorized_top_k/top_1_categorical_accuracy': [0.0017374999588355422,
  0.0013000000035390258,
  0.0006874999962747097,
  0.0006249999860301614,
  0.0007249999907799065,
  0.0008500000112690032,
  0.0025500000920146704,
  0.002062499988824129,
  0.0032999999821186066,
  0.0017625000327825546,
  0.0026374999433755875,
  0.001962499925866723,
  0.0032625000458210707,
  0.0033499998

In [53]:
accuracy = hist.history["factorized_top_k/top_100_categorical_accuracy"][-1]
print(f"Retrieval top-100 accuracy: {accuracy:.4f}.")

Retrieval top-100 accuracy: 0.1067.


In [54]:
root_mean_squared_error = hist.history["root_mean_squared_error"][-1]
print(f"Ranking RMSE: {root_mean_squared_error:.4f}.")

Ranking RMSE: 1.4672.
